<a href="https://colab.research.google.com/github/Abhinavks2006/GenerativeAI_Internship/blob/main/day_7_ls.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd

df = pd.read_csv("/content/IMDB Dataset.csv",encoding='latin1')

print(df.head())
print(df['sentiment'].value_counts())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [3]:
df['sentiment']=df['sentiment'].map({
    'positive':1,
    'negative':0
})

In [4]:
negation_words = [
    "not good", "not bad", "not great", "don't like "
    "didn't like", "never liked", "wasn't good",
    "isn't good", "no good"
]

In [5]:
import re

def clean_text(text):
  text = text.lower()

  text = re.sub(r"[^a-zA-Z\s']"," ", text)

  for phrase in negation_words:
    text = text.replace(phrase, phrase.replace(" ", "_")) # Changed to underscore for clarity

  return text

In [6]:
df['review']=df['review'].apply(clean_text)

In [7]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(df['review'],df['sentiment'],test_size=0.2,random_state=42)

In [8]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size = 20000
max_len = 250

tokenizer = Tokenizer(num_words=vocab_size,oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq,maxlen=max_len,padding='post')
X_test_pad = pad_sequences(X_test_seq,maxlen=max_len,padding='post')

In [18]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,Dropout
model = Sequential([
    Embedding(vocab_size,128,input_length=max_len),
    LSTM(128,dropout=0.3,recurrent_dropout=0.3),
    Dense(64,activation='relu'),
    Dropout(0.3),
    Dense(1,activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [20]:
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [21]:
history = model.fit(X_train_pad,y_train,batch_size=64,epochs=5,validation_split=0.2)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 421s 834ms/step - accuracy: 0.5461 - loss: 0.6713 - val_accuracy: 0.5639 - val_loss: 0.6609
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 416s 833ms/step - accuracy: 0.6017 - loss: 0.6112 - val_accuracy: 0.6070 - val_loss: 0.6527
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 441s 830ms/step - accuracy: 0.7777 - loss: 0.4687 - val_accuracy: 0.8434 - val_loss: 0.3884
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 530s 1s/step - accuracy: 0.8804 - loss: 0.3124 - val_accuracy: 0.8369 - val_loss: 0.4150
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 479s 842ms/step - accuracy: 0.9143 - loss: 0.2367 - val_accuracy: 0.8764 - val_loss: 0.3622


In [22]:
loss,accuracy = model.evaluate(X_test_pad,y_test)
print("Test Accuracy:",accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 32s 102ms/step - accuracy: 0.8850 - loss: 0.3439
Test Accuracy: 0.8849999904632568


In [23]:
def predict_sentiment(review):
  review = clean_text(review)
  sequence = tokenizer.texts_to_sequences([review])
  padded_sequence = pad_sequences(sequence,maxlen=max_len,padding='post')
  prediction = model.predict(padded_sequence)[0][0]
  print("\nReview:",review)
  print("Score:",prediction)

  if prediction >=0.5:
    print("Sentiment: Positive 😊")
  else:
    print("Sentiment: Negative 😢")

In [27]:
predict_sentiment("good movie")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step

Review: good movie
Score: 0.65498894
Sentiment: Positive 😊
